In [77]:
#import pandas for data storing object and numpy for numerical operations
import pandas as pd
import numpy as np

In [78]:
#read in the downloaded data file
raw_data=pd.read_csv('raw_data.csv')
df=raw_data.copy()

In [79]:
#print summary of the data
print(raw_data.columns,raw_data.shape)

#print each column
for col in raw_data.columns:
    print(col,raw_data[col])

Index(['Rank', 'Peak', 'All Time Peak', 'Actual gross',
       'Adjusted gross (in 2022 dollars)', 'Artist', 'Tour title', 'Year(s)',
       'Shows', 'Average gross', 'Ref.'],
      dtype='str') (20, 11)
Rank 0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      7
8      9
9     10
10    11
11    12
12    13
13    14
14    15
15    16
16    17
17    18
18    19
19    20
Name: Rank, dtype: int64
Peak 0         1
1         1
2      1[4]
3      2[7]
4      2[4]
5      2[4]
6     2[10]
7       NaN
8       NaN
9       NaN
10      NaN
11      NaN
12      NaN
13    1[20]
14     2[c]
15      NaN
16      NaN
17      NaN
18      NaN
19      NaN
Name: Peak, dtype: str
All Time Peak 0          2
1       7[2]
2       2[5]
3      10[7]
4        NaN
5      10[9]
6        NaN
7        NaN
8        NaN
9        NaN
10       NaN
11    14[17]
12       NaN
13       NaN
14       NaN
15       NaN
16       NaN
17       NaN
18       NaN
19       NaN
Name: All Time Peak, dtype: str
Actual gross

In [80]:
#loop through the columns and process data in column
from numpy import int16

#loop through the columns and conditionally process them one by one
for col in df.columns:

    #process all text columns remove references from all columns as this is a recurring theme in the data and prevents numerical analysis, same for commas and dollar signs
    if df[col].dtype=='str':

        #remove the reference boxes
        df[col]=df[col].str.replace(r'\[.*\]','',regex=True)

        #remove the commas and dollar signs from the cash entries
        df[col]=df[col].str.replace(',','',regex=True)
        df[col]=df[col].str.replace(r'\$','',regex=True)

        #remove the symbols in the tour names
        df[col]=df[col].str.replace(r'[†‡*]', '', regex=True)

        #remove the leading and trailing empty spaces
        df[col]=df[col].str.strip()
    
    #process the peak columns as these are full of NaN values and are in strings (use Int64 as this allows NaN values to be stored)
    if col in ['Peak', 'All Time Peak']:
        df[col]=pd.to_numeric(df[col], errors='coerce').astype('Int64')

    #process the ranking and shows columns to be integers as they are currently strings
    if col in ['Rank', 'shows']:
        df[col]=df[col].astype(int)

    #only drop NaN values in the key fields of interest (income) so we dont unnecessarily lose data in other fields
    if col in ['Actual gross', 'Shows']:
        df=df.dropna(subset=[col])

    #process the years to make them numerical (need separating into year_start and year_end)
    if col=='Year(s)':
        years=df[col].str.split('–', expand=True)

        #append the start and end years
        df['year_start']=years[0].astype(int)
        df['year_end']=years[1].fillna(years[0]).astype(int)

        #the original column is now redundant
        df=df.drop(columns='Year(s)')




In [81]:
#print summary of the data
print(df.columns,df.shape)

#print each column
for col in df.columns:
    print(col,df[col])

Index(['Rank', 'Peak', 'All Time Peak', 'Actual gross',
       'Adjusted gross (in 2022 dollars)', 'Artist', 'Tour title', 'Shows',
       'Average gross', 'Ref.', 'year_start', 'year_end'],
      dtype='str') (20, 12)
Rank 0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      7
8      9
9     10
10    11
11    12
12    13
13    14
14    15
15    16
16    17
17    18
18    19
19    20
Name: Rank, dtype: int64
Peak 0        1
1        1
2        1
3        2
4        2
5        2
6        2
7     <NA>
8     <NA>
9     <NA>
10    <NA>
11    <NA>
12    <NA>
13       1
14       2
15    <NA>
16    <NA>
17    <NA>
18    <NA>
19    <NA>
Name: Peak, dtype: Int64
All Time Peak 0        2
1        7
2        2
3       10
4     <NA>
5       10
6     <NA>
7     <NA>
8     <NA>
9     <NA>
10    <NA>
11      14
12    <NA>
13    <NA>
14    <NA>
15    <NA>
16    <NA>
17    <NA>
18    <NA>
19    <NA>
Name: All Time Peak, dtype: Int64
Actual gross 0     780000000
1     579800000
2     41